# Fixed-budget oracle

## Objective

Let $v_i\ge0$ be row importance and $M\in\{0,1\}^N$ a selection mask.
For its maximal contiguous chunks $\mathcal C(M)$, define

$$I(M)=\sum_{i=0}^{N-1}v_iM_i,\qquad
L(M)=\sum_{C\in\mathcal C(M)}T(|C|).$$

For $1\le R\le N$, select exactly $R$ rows:

$$\boxed{\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}.}$$

Latency is an additive model estimate.

In [1]:
from dataclasses   import dataclass
from pathlib       import Path
from scipy.ndimage import maximum_filter1d
from tqdm.auto     import tqdm

import json

import numpy  as np
import pandas as pd

/home/yabsed/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id="cost-model"></a>
## Latency model

For a chunk of $\ell$ rows,

$$T(0)=0,\qquad T(\ell)=b\ell+\delta\max(s,\ell)\quad(\ell\ge1).$$

Here $s$ is the boundary in rows, $b$ the short slope, and $\delta$ the slope change (ms per row).
Short chunks have intercept $\delta s$; long chunks have slope $b+\delta$.

Assume $s,b,\delta\ge0$ and $b+\delta>0$.
The boundary remains real-valued, and the model extends linearly beyond the measured range.

In [2]:
@dataclass(frozen=True)
class TwoLineCost:

    """Two-line latency model in row units."""

    boundary     : float
    short_slope  : float
    slope_change : float

    def table(self, n):
        
        """Return the latency for each run length from 0 through n."""

        lengths = np.arange(n + 1)
        latency = (
              self.short_slope  * lengths
            + self.slope_change * np.maximum(self.boundary, lengths)
        )
        latency[0] = 0
        return latency

<a id="solver"></a>
## Solver

For a fixed ratio $q$, solve the additive problem

$$F_R(q)=\max_{\|M\|_1=R}\{I(M)-qL(M)\}.$$

Dinkelbach iteration updates $q$ using the resulting mask.

<a id="bounds"></a>
### Ratio bounds

Every feasible mask satisfies

$$L(M)=bR+\delta\sum_C\max(s,|C|)\ge T(R).$$

Let $M_C$ be the most important contiguous chunk of length $R$, and $M_T$ select the $R$ largest values. Initially,

$$\ell_0=\frac{I(M_C)}{T(R)}
\le\eta_R^\star\le
u_0=\frac{I(M_T)}{T(R)}.$$

At $q=\ell$, let $M_q$ maximize $I-qL$. The residual $F_R(q)$ is nonnegative.
Update

$$\ell\leftarrow\max\left(\ell,\frac{I(M_q)}{L(M_q)}\right),\qquad
u\leftarrow\min\left(u,q+\frac{F_R(q)}{T(R)}\right).$$

The lower bound is a feasible ratio. For an optimal mask $M^\star$,

$$I(M^\star)-qL(M^\star)\le F_R(q),\qquad L(M^\star)\ge T(R),$$

which gives the upper bound.

Stop when $u-\ell\le\epsilon\ell_0$. For nonzero importance, exact arithmetic gives

$$0\le1-\frac{\ell}{\eta_R^\star}\le\epsilon.$$

The implementation uses float64 with $\epsilon=10^{-6}$ (`rtol`).
Zero importance returns immediately.

<a id="recurrence"></a>
### Bellman recurrence

Let $P_j=\sum_{i<j}v_i$. Define $D_{z,r}$ as the maximum $I-qL$ over
the first $z+r$ rows, with $z$ excluded and $r$ selected.

Place the last zero after a prefix containing $t$ selected rows.
The final chunk is $[z+t,z+r)$, with length $r-t$:

$$\boxed{D_{z,r}=\max_{0\le t\le r}
\left[D_{z-1,t}+P_{z+r}-P_{z+t}-qT(r-t)\right].}$$

At $t=r$, the final chunk is empty. With no zeros, every row is selected:

$$D_{0,r}=P_r-qT(r).$$

The terminal value is $F_R(q)=D_{N-R,R}$.

<a id="range-maxima"></a>
### Range maxima

The final chunk is empty, short, or long according to $r-t$.
Use the integer boundary $\bar s=\min(\lfloor s\rfloor,R)$ (`boundary` in code) for this split;
the cost intercept $a=\delta s$ still uses the real boundary.

Collect the terms depending on $t$:

$$A_{z,t}=D_{z-1,t}-P_{z+t}+qbt,\qquad
B_{z,t}=D_{z-1,t}-P_{z+t}+q(b+\delta)t.$$

Then

$$D_{z,r}=\max\left\{\begin{aligned}
&D_{z-1,r},\\
&P_{z+r}-q(a+br)+\max_{\max(0,r-\bar s)\le t<r}A_{z,t},\\
&P_{z+r}-q(b+\delta)r+\max_{0\le t<r-\bar s}B_{z,t}.
\end{aligned}\right.$$

The short-chunk candidates form a sliding window; the long-chunk candidates form a growing prefix.
Both maxima take $O(R+1)$ time per DP row. An empty maximum is $-\infty$.

<a id="recovery"></a>
### Recovery

Start from $(N-R,R)$. Choose a maximizing $t$, select the final chunk
$[z+t,z+r)$, and continue from $(z-1,t)$.
Each step removes one zero from the unresolved prefix.
At $z=0$, select the remaining prefix.

One additive solve, including recovery, takes $O((N-R+1)(R+1))$ time
and memory. For $K$ solves, multiply the time by $K$.

### Implementation

In [3]:
def chunks(mask):
    """Return selected chunks as [start, end) intervals."""

    padded = np.concatenate(([0], mask, [0]))
    change = np.diff(padded)

    starts = np.flatnonzero(change ==  1) # 0 -> 1
    ends   = np.flatnonzero(change == -1) # 1 -> 0

    return np.column_stack((starts, ends))

In [4]:
def trailing_max(values, width):

    """Return the maximum of the last width values at each position.

    Example: [-1, -2, -3, -4, -5] with width=3 gives [-1, -1, -1, -2, -3].
    """
    if width < 1:
        raise ValueError("width must be positive")

    return maximum_filter1d(
        values,
        size   = width,

        # maximize with indices [j - width + 1, j]
        origin = (width - 1) // 2,

        # assign -inf to invaild indices
        mode   = "constant",
        cval   = -np.inf,
    )


In [5]:
class SelectionProblem:

    """One trace and its cost model, prepared once for all row budgets.

    Create a new problem when the importance values or cost model change.
    R, q, DP, and masks belong to individual calculations.
    """

    def __init__(self, v, cost: TwoLineCost):

        # Importance
        self.v = np.array(v, dtype=float, copy=True)
        self.P = np.concatenate(([0.0], np.cumsum(self.v)))

        # Latency
        self.cost = cost
        self.T    = cost.table(len(self.v))

    def solve(self, R, rtol=1e-6):

        """Return the best mask and ratio bounds."""

        T = self.T

        # Ratio bounds
        mask  = self.best_run(R)
        lower = self.total_importance(mask) / T[R]
        upper = self.total_importance(self.top_r(R)) / T[R]

        # Fixed tolerance
        tolerance = rtol * lower

        while upper - lower > tolerance:

        # 1. MAXIMIZATION: I - qL
        
            q  = lower
            DP = self.bellman(R, q)

        # 2. LOWER BOUND

            # recover best mask
            candidate       = self.recover(R, q, DP)
            importance      = self.total_importance(candidate)
            latency         = self.total_latency(candidate)
            candidate_ratio = importance / latency

            # update
            if candidate_ratio > lower:
                mask  = candidate
                lower = candidate_ratio

        # 3. UPPER BOUND

            # residual: best value of I - qL
            residual = max(0.0, DP[-1, -1])
            upper    = min(upper, q + residual / T[R])

            # roundoff
            upper = max(upper, lower)

        # 4. STAGNATION

            if lower <= q and upper - lower > tolerance:
                raise RuntimeError("Dinkelbach iteration stalled.")

        # GOTO WHILE
            
        return mask, lower, upper

    def bellman(self, R, q):

        """Compute the best importance - q * latency for each DP state."""

        # Data
        P    = self.P
        T    = self.T
        cost = self.cost

        # Latency model
        short_slope  = cost.short_slope
        slope_change = cost.slope_change
        intercept    = slope_change * cost.boundary

        # DP boundary
        boundary = min(int(np.floor(cost.boundary)), R)

        # Row counts
        Z = len(self.v) - R  # Unselected rows

        t = np.arange(R + 1)  # Previous selected count
        r = np.arange(R + 1)  # Current selected count

        # DP[z, r]: best I - qL over the first z + r rows
        # z excluded, r selected
        DP    = np.empty((Z + 1, R + 1))
        DP[0] = P[:R + 1] - q * T[:R + 1]

        for z in range(1, Z + 1):

            # Last zero at z + t - 1; final chunk length r - t.
            A = DP[z - 1, t] - P[z + t] + q * short_slope * t
            B = DP[z - 1, t] - P[z + t] + q * (short_slope + slope_change) * t

            # Short chunk: length <= boundary
            short_max = np.full(R + 1, -np.inf)
            if boundary:
                short_max[1:] = trailing_max(A[:-1], boundary)

            # Long chunk: length > boundary
            long_max = np.full(R + 1, -np.inf)
            long_max[boundary + 1:] = np.maximum.accumulate(B)[:R - boundary]

            # Choose the best empty, short, or long final chunk.
            DP[z] = np.maximum.reduce([
                DP[z - 1],
                P[z + r] - q * (intercept + short_slope * r) + short_max,
                P[z + r] - q * (short_slope + slope_change) * r + long_max,
            ])

        return DP

    def recover(self, R, q, DP):
        """Recover a mask maximizing I - qL."""

        P = self.P
        T = self.T
        Z = len(self.v) - R

        mask = np.zeros(len(self.v), dtype=bool)
        r = R

        for z in range(Z, 0, -1):

            # Last zero
            t = np.arange(r + 1)
            scores = (
                DP[z - 1, t]           # Prefix score
                + P[z + r] - P[z + t]  # Chunk importance
                - q * T[r - t]         # Latency penalty
            )
            best_t = int(np.argmax(scores))

            # Final chunk
            mask[z + best_t:z + r] = True

            # Previous state
            r = best_t

        # Remaining prefix: all selected
        mask[:r] = True
        return mask

    def best_run(self, R):
        """Select R consecutive rows with maximum importance."""

        P = self.P

        # Chunk importance
        importance = P[R:] - P[:-R]

        # Best chunk
        start                 = int(np.argmax(importance))
        mask                  = np.zeros(len(self.v), dtype=bool)
        mask[start:start + R] = True

        return mask

    def top_r(self, R):
        """Select the R rows with greatest importance."""

        # Importance ranking
        order = np.argsort(-self.v, kind="stable")

        # Top R rows
        mask            = np.zeros(len(self.v), dtype=bool)
        mask[order[:R]] = True

        return mask

    def total_importance(self, mask):

        """Return the total importance of selected rows."""
        return self.v[mask].sum()

    def total_latency(self, mask):

        """Return the total latency of selected runs."""
        spans   = chunks(mask)
        lengths = spans[:, 1] - spans[:, 0]
        return self.T[lengths].sum()

<a id="inputs"></a>
## Inputs

Use all traces from [02_01](02_01_capture_traces.ipynb) in saved order,
with their captured importance values.
Load the `Two-line (free)` fit from [experiment 01](../01/01_02_chunk_latency.ipynb).

In [6]:
MODEL = Path("../01/latency_model.json")
TRACE = Path(  "activation_traces.json")

In [7]:
traces = json.loads(TRACE.read_text())["traces"]
for trace in traces:
    trace["values"] = np.asarray(trace["values"], dtype=float)

In [8]:
model_data = json.loads(MODEL.read_text())
model      = model_data["models"]["Two-line (free)"]

# The experiment uses the two-line fit with a zero tail intercept.
boundary_kib     = model["boundary_kib"]
short_slope_kib  = model["short_slope_ms_per_kib"]
slope_change_kib = model["slope_change_ms_per_kib"]

<a id="row-units"></a>
### Row units

Let $w=\mathrm{row\_bytes}/1024$ be the size of one weight row in KiB.
A chunk of $\ell$ rows occupies $w\ell$ KiB, so

$$s=\frac{s_{\mathrm{KiB}}}{w},\qquad
b=wb_{\mathrm{KiB}},\qquad
\delta=w\delta_{\mathrm{KiB}}.$$

In [9]:
cases = []

for trace in traces:
    
    # Importance
    v = trace["values"]
    
    # KiB per row
    w = trace["row_bytes"] / 1024

    # Cost model: KiB -> rows
    cost = TwoLineCost(
        boundary     = boundary_kib / w, 
        short_slope  = w * short_slope_kib, 
        slope_change = w * slope_change_kib
    )

    # Selection problem
    problem = SelectionProblem(v, cost)
    cases.append((trace, problem))

<a id="run"></a>
## Reference

For each trace and fraction $f\in\{0.1,\ldots,0.9\}$, select
$R=\max(1,\lfloor fN\rfloor)$ rows.

Save the mask, importance, latency, ratio bounds, and model inputs to
`oracle_reference.csv` for [02_03](02_03_selector_sweep.ipynb).
Masks use half-open intervals $[\mathrm{start},\mathrm{end})$.
[02_04](02_04_analyze_oracle.ipynb) computes ratio gaps and fragmentation.

In [10]:
# Requested fractions of rows, from 0.1 through 0.9.
BUDGETS = np.arange(1, 10) / 10

records = []
for trace, problem in tqdm(cases, desc="Oracle reference", unit="trace"):
    for fraction in BUDGETS:

        # decide R using fraction
        R = max(1, int(fraction * len(problem.v)))

        # optimal mask & its (importance)/(latency) ratio
        mask, lower, upper = problem.solve(R)

        records.append({
            # rewrite trace infomation
            "trace_id" : trace["trace_id"],
            "n"        : trace["n"],
            "d"        : trace["d"],

            # fraction
            "fraction"  : fraction,
            "R_nominal" : R,

            # total importance
            "importance_total" : problem.v.sum(),

            # latency model
            "saturation_rows"          : problem.cost.boundary,
            "short_slope_ms_per_row"   : problem.cost.short_slope,
            "slope_change_ms_per_row"  : problem.cost.slope_change,

            # optimal mask
            "importance" : problem.total_importance(mask),
            "latency"    : problem.total_latency   (mask),
            "eta_lower"  : lower,
            "eta_upper"  : upper,
            "intervals"  : json.dumps(chunks(mask).tolist()),
        })

Oracle reference: 100%|██████████| 504/504 [03:28<00:00,  2.42trace/s]


In [11]:
reference = pd.DataFrame(records)
assert not reference.duplicated(["trace_id", "fraction"]).any()
reference.head()

,trace_id,n,d,fraction,R_nominal,importance_total,saturation_rows,short_slope_ms_per_row,slope_change_ms_per_row,importance,latency,eta_lower,eta_upper,intervals
0,0,896,896,0.1,89,52.812195,118.857143,0.000219,0.000103,7.156982,0.031700,225.769502,225.769502,"[[483, 572]]"
1,0,896,896,0.2,179,52.812195,118.857143,0.000219,0.000103,14.249634,0.063620,223.981605,223.981605,"[[13, 86], [481, 587]]"
2,0,896,896,0.3,268,52.812195,118.857143,0.000219,0.000103,20.660400,0.095320,216.747673,216.747673,"[[35, 86], [483, 585], [670, 785]]"
3,0,896,896,0.4,358,52.812195,118.857143,0.000219,0.000103,25.059204,0.115165,217.593337,217.593337,"[[8, 127], [483, 603], [670, 789]]"
4,0,896,896,0.5,448,52.812195,118.857143,0.000219,0.000103,30.990967,0.146967,210.870145,210.870145,"[[13, 108], [313, 432], [483, 602], [670, 785]]"


In [12]:
RESULTS = Path("oracle_reference.csv")
reference.to_csv(RESULTS, index=False)